# Vicuna Standalone Diagnosis

**Problem:** Vicuna-7B and Vicuna-13B (standalone LLM via `backbone_nothink`) produce near-100% empty outputs.  
**Root cause hypothesis:** The chat template causes the model to immediately generate `</s>` (EOS) before producing any answer tokens.

This notebook:
1. Measures empty-output rates per model / condition / variant
2. Inspects the first-token logits to confirm the EOS-collapse pattern
3. Compares with LLaVA-Vicuna VLM and LM-decoder (same backbone, different prompt wrapper)
4. Shows what answer Vicuna *would have* given (second-best token analysis)
5. Checks whether the issue is prompt-format-dependent (variant C vs B vs A)

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BASE = Path('../../')   # repo root
sys.path.insert(0, str(BASE))
sys.path.insert(0, str(BASE / 'analysis'))

LOGITS_BN  = BASE / 'evaluation/logits/backbone_nothink/pretrained'
LOGITS_LMD = BASE / 'evaluation/logits/lm_decoder/pretrained'
RESULTS_PT = BASE / 'evaluation/results/pretrained'

VICUNA_MODELS = {
    'Vicuna-7B  (standalone)':  LOGITS_BN  / 'vicuna-7b-v1.5',
    'Vicuna-13B (standalone)':  LOGITS_BN  / 'vicuna-13b-v1.5',
    'LLaVA-Vicuna (LM decoder)': LOGITS_LMD / 'llava-v1.6-vicuna-7b-hf',
}

CONDITIONS = ['vqa_1k_control_blind', 'vqa_1k_control_inst_blind']
CT_TO_VARIANT = {'question': 'C', 'weaker_object': 'B', 'pronominalized': 'A'}

plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False})

## 1. Load all Vicuna outputs

In [ ]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(l) for l in path.open() if l.strip()]


rows = []
for model_label, model_dir in VICUNA_MODELS.items():
    for cond in CONDITIONS:
        cond_short = 'blind' if cond.endswith('blind') and 'inst' not in cond else 'inst_blind'
        records = load_jsonl(model_dir / f'{cond}.jsonl')
        for ex in records:
            ga = ex.get('generated_answers', {})
            gl = ex.get('generated_logits', {})
            for ct, var in CT_TO_VARIANT.items():
                ans = str(ga.get(ct, ''))
                logit_info = gl.get(ct, {}).get('content', [])
                n_tokens = len(logit_info)
                first_token = logit_info[0]['token'] if logit_info else None
                first_logp  = logit_info[0]['logprob'] if logit_info else None
                top_tokens  = [(t['token'], round(t['logprob'], 3))
                               for t in (logit_info[0].get('top_logprobs', []) if logit_info else [])]
                rows.append(dict(
                    model=model_label, condition=cond_short, variant=var,
                    question_id=ex['question_id'],
                    answer=ans,
                    is_empty=ans.strip() == '',
                    n_tokens=n_tokens,
                    first_token=first_token,
                    first_logp=first_logp,
                    top_tokens=top_tokens,
                ))

df = pd.DataFrame(rows)
print(f'Total rows: {len(df)}')
df.head(3)

## 2. Empty-output rate by model × condition × variant

In [ ]:
empty_rate = (df.groupby(['model', 'condition', 'variant'])['is_empty']
               .mean()
               .rename('empty_rate')
               .reset_index())

pivot = empty_rate.pivot_table(
    index='model', columns=['condition', 'variant'], values='empty_rate'
)
print('Empty output rate (fraction):')
pivot.style.format('{:.1%}').background_gradient(cmap='Reds', vmin=0, vmax=1)

## 3. First-token distribution — EOS collapse confirmation

In [ ]:
# Focus on standalone models, variant C, blind condition
sub = df[(df['condition'] == 'blind') & (df['variant'] == 'C')].copy()

fig, axes = plt.subplots(1, len(VICUNA_MODELS), figsize=(14, 3.5), sharey=False)

for ax, (model_label, _) in zip(axes, VICUNA_MODELS.items()):
    m_sub = sub[sub['model'] == model_label]
    tok_counts = Counter(m_sub['first_token'].dropna())
    top10 = tok_counts.most_common(10)
    tokens, counts = zip(*top10) if top10 else ([], [])
    tokens = [repr(t) for t in tokens]
    colors = ['#E53935' if '</s>' in t or t == "''" else '#5C6BC0' for t in tokens]
    ax.barh(range(len(tokens)), counts, color=colors, edgecolor='none', height=0.7)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('# questions', fontsize=9)
    ax.set_title(model_label.replace(' (', '\n('), fontsize=9)

fig.suptitle('First-token distribution — blind, variant C', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 4. EOS logprob distribution (standalone Vicuna)

If `</s>` has a very high logprob (close to 0), the model is confidently ending generation immediately — not struggling, just formatted wrong.

In [ ]:
standalone = df[df['model'].str.contains('standalone') & (df['variant'] == 'C')].copy()

fig, ax = plt.subplots(figsize=(7, 3.5))
for model_label in standalone['model'].unique():
    m = standalone[standalone['model'] == model_label]
    for cond, ls in [('blind', '-'), ('inst_blind', '--')]:
        vals = m[m['condition'] == cond]['first_logp'].dropna()
        if vals.empty:
            continue
        ax.hist(vals, bins=40, histtype='step', lw=1.5, ls=ls,
                label=f'{model_label.split(" ")[1]} / {cond}')

ax.set_xlabel('First-token log-probability', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_title('First-token logprob — standalone Vicuna (variant C)', fontsize=11)
ax.axvline(-1.0, color='gray', lw=1, ls=':', alpha=0.5, label='logp = -1')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print('Median first-token logprob:')
print(standalone.groupby(['model', 'condition'])['first_logp'].median().round(3))

## 5. What would Vicuna have said? — 2nd-best token analysis

Since top-1 is always `</s>`, we look at top-2 to see what the model *wanted* to say.

In [ ]:
def get_second_best(top_tokens):
    """Return the 2nd-best token (skipping </s> and whitespace-only tokens)."""
    if not top_tokens:
        return None, None
    for tok, lp in top_tokens[1:]:  # skip rank-1 (always </s>)
        if tok.strip() and tok not in ('</s>', '<eos>', '<|endoftext|>'):
            return tok, lp
    return None, None


v7b = df[(df['model'] == 'Vicuna-7B  (standalone)') &
         (df['condition'] == 'blind') &
         (df['variant'] == 'C')].copy()

v7b[['second_token', 'second_logp']] = v7b['top_tokens'].apply(
    lambda x: pd.Series(get_second_best(x))
)

print('Top 20 second-best tokens (what Vicuna-7B wanted to say):')
print(v7b['second_token'].value_counts().head(20))

fig, ax = plt.subplots(figsize=(8, 3.5))
top20 = v7b['second_token'].value_counts().head(20)
ax.barh(range(len(top20)), top20.values, color='#5C6BC0', height=0.7)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([repr(t) for t in top20.index], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('# questions', fontsize=9)
ax.set_title('Vicuna-7B (standalone): 2nd-best first token — blind, variant C', fontsize=10)
plt.tight_layout()
plt.show()

## 6. EOS probability vs. logprob gap to 2nd token

A large gap (EOS logp >> 2nd-best logp) means the model is *certain* it should stop — genuine template confusion, not low-confidence guessing.

In [ ]:
def logp_gap(top_tokens):
    """Gap between rank-1 and rank-2 logprob."""
    if len(top_tokens) < 2:
        return np.nan
    return top_tokens[0][1] - top_tokens[1][1]


v7b['logp_gap'] = v7b['top_tokens'].apply(logp_gap)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(v7b['first_logp'].dropna(), bins=30, color='#E53935', edgecolor='none')
axes[0].set_xlabel('EOS log-probability', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].set_title('EOS logprob (rank-1)', fontsize=10)

axes[1].hist(v7b['logp_gap'].dropna(), bins=30, color='#5C6BC0', edgecolor='none')
axes[1].set_xlabel('Logprob gap (rank-1 − rank-2)', fontsize=10)
axes[1].set_title('Confidence gap: EOS vs next best', fontsize=10)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Vicuna-7B standalone — blind, variant C', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Mean EOS logprob:  {v7b["first_logp"].mean():.3f}')
print(f'Mean logprob gap:  {v7b["logp_gap"].mean():.3f}')
print(f'Frac with gap > 1: {(v7b["logp_gap"] > 1).mean():.1%}')

## 7. Compare prompt format: standalone vs LM-decoder vs VLM

LLaVA-Vicuna uses a different chat template wrapper, which evidently doesn't trigger EOS collapse. Let's look at the actual prompt strings.

In [ ]:
# Print example question prompts for each inference mode
example_qid = None

print('=== Standalone Vicuna-7B prompt ===')
records_bn = load_jsonl(LOGITS_BN / 'vicuna-7b-v1.5' / 'vqa_1k_control_blind.jsonl')
ex = records_bn[0]
example_qid = ex['question_id']
print(f'  question field: {repr(ex["question"])}')
print()

print('=== LLaVA-Vicuna LM-decoder prompt ===')
records_lm = load_jsonl(LOGITS_LMD / 'llava-v1.6-vicuna-7b-hf' / 'vqa_1k_control_blind.jsonl')
for ex in records_lm:
    if ex['question_id'] == example_qid:
        print(f'  question field: {repr(ex["question"])}')
        ga = ex.get('generated_answers', {})
        print(f'  generated answer (C): {repr(ga.get("question",""))}')
        break

print()
print('=== LLaVA-Vicuna VLM output ===')
records_vlm = load_jsonl(RESULTS_PT / 'llava-v1.6-vicuna-7b-hf' / 'vqa_1k_blind.jsonl')
for ex in records_vlm:
    if ex['question_id'] == example_qid:
        print(f'  question prompt: {repr(ex.get("question",""))[:200]}')
        print(f'  output: {repr(ex.get("output",""))}')
        break

## 8. Does the variant matter? EOS rate by variant

In [ ]:
variant_eos = (df[df['model'].str.contains('standalone')]
               .groupby(['model', 'condition', 'variant'])['is_empty']
               .mean()
               .rename('empty_rate')
               .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
cond_titles = {'blind': 'Blind', 'inst_blind': 'Inst-Blind'}

x = np.arange(3)
variants = ['C', 'B', 'A']
colors = {'Vicuna-7B  (standalone)': '#E53935', 'Vicuna-13B (standalone)': '#C62828'}
width = 0.35

for ax, cond in zip(axes, ['blind', 'inst_blind']):
    sub = variant_eos[variant_eos['condition'] == cond]
    for i, model in enumerate(colors):
        m = sub[sub['model'] == model].set_index('variant')['empty_rate']
        vals = [m.get(v, 0) for v in variants]
        ax.bar(x + i * width, vals, width, label=model.split('(')[0].strip(),
               color=list(colors.values())[i], alpha=0.85)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(['C (original)', 'B (weaker)', 'A (pronoun.)'])
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(cond_titles[cond], fontsize=11)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Empty output rate', fontsize=10)
plt.suptitle('Empty output rate by variant — standalone Vicuna', fontsize=11)
plt.tight_layout()
plt.show()

## 9. Summary and conclusions

In [ ]:
print('=== DIAGNOSIS SUMMARY ===')
print()
for model in df['model'].unique():
    for cond in ['blind', 'inst_blind']:
        sub = df[(df['model'] == model) & (df['condition'] == cond) & (df['variant'] == 'C')]
        if sub.empty:
            continue
        er = sub['is_empty'].mean()
        n_tok_med = sub['n_tokens'].median()
        print(f'{model:<35} [{cond:<10}]  empty={er:.1%}  median_n_tokens={n_tok_med}')